In [9]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

# Tutorial
In this short tutorial I will show you how to use implemented CDEF metric on your own examples. Let's suppose that we have a sentence *"Mary had a little lamb, its fleece was white as snow."* with entities: *Mary* of type *person* and *little lamb* of type *animal*. We want to check how generated entities *mary* of type *person*, *lamb* of type *animal* and *snow* of type *place* corresponds to the correct entities.

In [ ]:
sentence = "Mary had a little lamb, its fleece was white as snow."

types = ['person', 'animal', 'place']

gold_entities = [
    ["Mary", "person"],
    ["little lamb", "animal"]
]

generated_entities = [
    ['mary', 'person'], 
    ['lamb', 'animal'],
    ['snow', 'place']
]

Because metric *CDEF* operates on embedding vectors, first we need to calculate embeddings. To do so, let's use method *get_embeddings* that was created for the purpose of this project. Of course you can use other methods to obtain embeddings, just keep in mind that to run *CDEF* you will need embeddings of gold entities and generated entities and types of these entities.

In [12]:
from models.llm_text_embeddings import get_embeddings

gold_embeddings = get_embeddings(gold_entities)
generated_embeddings = get_embeddings(generated_entities)

In [15]:
print(
    f"Function get_embeddings returns a matrix, that contains pairs [embedding, type]. Each pair for each entity.\n"
    f"Therefore in case of gold_entities, there are {len(gold_embeddings)} pairs.\n"
    f"Each pair contain embedding {type(gold_embeddings[0][0])} of size {len(gold_embeddings[0][0])}"
    f" and a type {type(gold_embeddings[0][1])}"
)

Function get_embeddings returns a matrix, that contains pairs [embedding, type]. Each pair for each entity.
Therefore in case of gold_entities, there are 2 pairs.
Each pair contain embedding <class 'list'> of size 768 and a type <class 'str'>


In [ ]:
from util.parse_llm_response import parse_response
from models.llm_ner import get_entities_from_llm


response = get_entities_from_llm(sentence, types)
generated_entities = parse_response(response)
gold_embeddings = get_embeddings(gold_entities)
generated_embeddings = get_embeddings(generated_entities)

In [8]:
print(generated_entities)

[['mary', 'person'], ['lamb', 'animal']]


In [6]:
from models.metric import CDE, exhaustive_CDE, EF, CDEF

cde=CDE(gold_embeddings, generated_embeddings)
exh_cde=exhaustive_CDE(gold_embeddings, generated_embeddings)
ef=EF(gold_embeddings, generated_embeddings)
cdef_05=CDEF(gold_embeddings, generated_embeddings, beta=0.5)
cdef_1=CDEF(gold_embeddings, generated_embeddings, beta=1)
cdef_15=CDEF(gold_embeddings, generated_embeddings, beta=1.5)

In [7]:
print(f'{"CDE:":10s}{cde}')
print(f'{"Exh_CDE":10s}{exh_cde}')
print(f'{"EF:":10s}{ef}')
print(f'{"CDEF-0.5":10s}{cdef_05}')
print(f'{"CDEF-1.0:":10s}{cdef_1}')

CDE:      0.0694176196972901
Exh_CDE   0.0694176196972901
EF:       0.0
CDEF-0.5  0.9720388524906738
CDEF-1.0: 0.9823391006775073
